# v1 vs v2 Pipeline Comparison

Side-by-side evaluation of the **manual v1** pipeline vs the **LlamaIndex v2** pipeline.

## What changed?

| Component | v1 (manual) | v2 (LlamaIndex) |
|---|---|---|
| Ingestion | `html_loader.py` + `chunkers.py` | `SEC10KReader` + `SentenceSplitter` |
| Indexing | Raw `chromadb.Collection.add()` + manual OpenAI batching | `VectorStoreIndex` + `ChromaVectorStore` |
| Semantic retrieval | `semantic_search()` custom function | `VectorIndexRetriever` |
| BM25 retrieval | `BM25Retriever` class | `BM25LlamaRetriever` (rank_bm25 wrapped) |
| Hybrid RRF | Manual rank fusion loop | `QueryFusionRetriever(mode='reciprocal_rerank')` |
| Reranking | `cross_encoder.predict()` | `SentenceTransformerRerank` post-processor |
| Generation | Raw OpenAI API + regex citations | `RetrieverQueryEngine` + `PromptTemplate` |
| Retrieval eval | Custom R@k / MRR loops | `RetrieverEvaluator` (HitRate + MRR built-in) |
| Generation eval | RAGAS | LlamaIndex `FaithfulnessEvaluator` + `RelevancyEvaluator` |

## Evaluation metrics for comparison

| Metric | Description |
|---|---|
| **R@k / HitRate@k** | % of questions where any gold chunk is in top-k |
| **MRR** | Mean Reciprocal Rank of first gold hit |
| **Faithfulness** | Are all claims in the answer supported by retrieved context? |
| **Answer Relevancy / Relevancy** | Does the answer address the question? |
| **Latency (ms/q)** | Wall-clock time per query |

> **Prerequisites:** Run `scripts/run_v2_retrieval_eval.py` and `scripts/run_v2_generation_eval.py`
> to generate `data/eval/v2_*.json` before executing this notebook.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path().resolve().parent
EVAL = ROOT / 'data' / 'eval'

# ---------- Load v1 results ----------
v1_summary = json.loads((EVAL / 'retrieval_summary.json').read_text())
v1_ragas   = json.loads((EVAL / 'ragas_summary.json').read_text())

# ---------- Load v2 results (generated by run_v2_*.py) ----------
v2_retrieval_path = EVAL / 'v2_retrieval_summary.json'
v2_gen_path       = EVAL / 'v2_generation_summary.json'

if v2_retrieval_path.exists():
    v2_summary = json.loads(v2_retrieval_path.read_text())
else:
    print('WARNING: v2_retrieval_summary.json not found.')
    print('Run:  python scripts/run_v2_retrieval_eval.py')
    v2_summary = []

if v2_gen_path.exists():
    v2_gen_summary = json.loads(v2_gen_path.read_text())
else:
    print('WARNING: v2_generation_summary.json not found.')
    print('Run:  python scripts/run_v2_generation_eval.py')
    v2_gen_summary = {}

print('v1 strategies:', [s['strategy'] for s in v1_summary])
print('v2 strategies:', [s['strategy'] for s in v2_summary])
print('v1 RAGAS:', v1_ragas)
print('v2 Gen  :', v2_gen_summary)

## Retrieval Comparison — R@k and MRR

In [ ]:
# Build comparison DataFrame
rows = []
for s in v1_summary:
    rows.append({
        'pipeline': 'v1',
        'strategy': s['strategy'],
        'R@1': s.get('recall@1', s.get('primary_recall@1', 0)),
        'R@5': s.get('recall@5', s.get('primary_recall@5', 0)),
        'R@10': s.get('recall@10', s.get('primary_recall@10', 0)),
        'MRR': s.get('mrr', 0),
        'ms/q': s.get('mean_latency_ms', 0),
    })
for s in v2_summary:
    rows.append({
        'pipeline': 'v2',
        'strategy': s['strategy'],
        'R@1': s.get('recall@1', 0),
        'R@5': s.get('recall@5', 0),
        'R@10': s.get('recall@10', 0),
        'MRR': s.get('mrr', 0),
        'ms/q': s.get('mean_latency_ms', 0),
    })

df = pd.DataFrame(rows)
df['label'] = df['pipeline'] + '/' + df['strategy']
df.set_index('label')[['R@1','R@5','R@10','MRR','ms/q']].style \
    .highlight_max(axis=0, color='#c6efce', subset=['R@1','R@5','R@10','MRR']) \
    .highlight_min(axis=0, color='#c6efce', subset=['ms/q']) \
    .format({'R@1':'{:.3f}','R@5':'{:.3f}','R@10':'{:.3f}','MRR':'{:.4f}','ms/q':'{:.1f}'})

## Retrieval Bar Chart — R@5 by Strategy and Pipeline

In [ ]:
# Group by strategy, compare v1 vs v2 side by side
common_strategies = [s for s in ['semantic','bm25','hybrid']
                     if s in df['strategy'].values]

v1_r5 = [df[(df.pipeline=='v1') & (df.strategy==s)]['R@5'].values[0]
          if len(df[(df.pipeline=='v1') & (df.strategy==s)]) > 0 else 0
          for s in common_strategies]
v2_r5 = [df[(df.pipeline=='v2') & (df.strategy==s)]['R@5'].values[0]
          if len(df[(df.pipeline=='v2') & (df.strategy==s)]) > 0 else 0
          for s in common_strategies]

x = np.arange(len(common_strategies))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, v1_r5, w, label='v1 (manual)', color='#4472C4')
b2 = ax.bar(x + w/2, v2_r5, w, label='v2 (LlamaIndex)', color='#ED7D31')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(common_strategies, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Recall@5')
ax.set_title('v1 vs v2: Recall@5 by Retrieval Strategy\n(Apple 2025 10-K, 50 gold questions)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'v1_v2_recall5.png', dpi=150)
plt.show()

## MRR Comparison

In [ ]:
v1_mrr = [df[(df.pipeline=='v1') & (df.strategy==s)]['MRR'].values[0]
           if len(df[(df.pipeline=='v1') & (df.strategy==s)]) > 0 else 0
           for s in common_strategies]
v2_mrr = [df[(df.pipeline=='v2') & (df.strategy==s)]['MRR'].values[0]
           if len(df[(df.pipeline=='v2') & (df.strategy==s)]) > 0 else 0
           for s in common_strategies]

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, v1_mrr, w, label='v1 (manual)', color='#5B9BD5')
b2 = ax.bar(x + w/2, v2_mrr, w, label='v2 (LlamaIndex)', color='#FF7F50')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(common_strategies, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('MRR')
ax.set_title('v1 vs v2: Mean Reciprocal Rank')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'v1_v2_mrr.png', dpi=150)
plt.show()

## Latency Comparison (ms/query)

In [ ]:
v1_lat = [df[(df.pipeline=='v1') & (df.strategy==s)]['ms/q'].values[0]
           if len(df[(df.pipeline=='v1') & (df.strategy==s)]) > 0 else 0
           for s in common_strategies]
v2_lat = [df[(df.pipeline=='v2') & (df.strategy==s)]['ms/q'].values[0]
           if len(df[(df.pipeline=='v2') & (df.strategy==s)]) > 0 else 0
           for s in common_strategies]

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, v1_lat, w, label='v1 (manual)', color='#A9D18E')
b2 = ax.bar(x + w/2, v2_lat, w, label='v2 (LlamaIndex)', color='#FF6B6B')

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
            f'{h:.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(common_strategies, fontsize=11)
ax.set_ylabel('ms / query')
ax.set_title('v1 vs v2: Query Latency')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'v1_v2_latency.png', dpi=150)
plt.show()

## Generation Quality Comparison

**v1 metrics (RAGAS):** Faithfulness, Answer Relevancy, Context Recall, Context Precision  
**v2 metrics (LlamaIndex):** Faithfulness, Relevancy  

Both evaluators use GPT-4o-mini as the LLM judge on the same 20 gold questions.

### Metric alignment

| Concept | v1 (RAGAS name) | v2 (LlamaIndex name) | Same idea? |
|---|---|---|---|
| Hallucination check | Faithfulness | Faithfulness | Yes |
| On-topic answer | Answer Relevancy | Relevancy | Yes |
| Coverage of gold info | Context Recall | _(not implemented)_ | Partial |
| Signal-to-noise in context | Context Precision | _(not implemented)_ | Partial |

In [ ]:
gen_data = [
    ('v1 Faithfulness',      v1_ragas.get('faithfulness'),       '#4472C4'),
    ('v2 Faithfulness',      v2_gen_summary.get('faithfulness'), '#ED7D31'),
    ('v1 Answer Relevancy',  v1_ragas.get('answer_relevancy'),   '#5B9BD5'),
    ('v2 Relevancy',         v2_gen_summary.get('relevancy'),    '#FF7F50'),
]

labels = [d[0] for d in gen_data]
values = [d[1] if d[1] is not None else 0 for d in gen_data]
colors = [d[2] for d in gen_data]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor='white')
for bar in bars:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_ylabel('Score (0-1)')
ax.set_title('v1 (RAGAS) vs v2 (LlamaIndex) Generation Quality\nGPT-4o-mini + hybrid retrieval | 20 questions')
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='0.8 threshold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'v1_v2_generation.png', dpi=150)
plt.show()

print('\nNote: v1 uses RAGAS (NLI + cross-encoding); v2 uses LlamaIndex LLM-as-judge.')
print('Both measure the same concepts but with different implementations.')

## Code Complexity Comparison

In [ ]:
import subprocess
import os

def count_lines(directory: Path) -> dict:
    total = 0
    per_file = {}
    for f in sorted(directory.rglob('*.py')):
        if '__pycache__' in str(f):
            continue
        lines = len(f.read_text(encoding='utf-8', errors='ignore').splitlines())
        per_file[str(f.relative_to(ROOT))] = lines
        total += lines
    return {'total': total, 'per_file': per_file}

v1_stats = count_lines(ROOT / 'src')
v2_stats = count_lines(ROOT / 'v2')

print(f"v1 pipeline code (src/):  {v1_stats['total']:>4d} lines")
print(f"v2 pipeline code (v2/):   {v2_stats['total']:>4d} lines")
print(f"Reduction: {v1_stats['total'] - v2_stats['total']} lines ({(1 - v2_stats['total']/v1_stats['total'])*100:.0f}%)")
print()
print('v2 file breakdown:')
for fname, lc in v2_stats['per_file'].items():
    print(f'  {fname:<55s} {lc:>4d} lines')

## Key Takeaways

### What LlamaIndex buys you

1. **`QueryFusionRetriever`** replaces ~40 lines of manual RRF code with one constructor call.
2. **`RetrieverQueryEngine`** handles the retrieve → format → generate loop automatically.
3. **`VectorStoreIndex`** manages batched embedding, storage, and reload from ChromaDB.
4. **`FaithfulnessEvaluator` / `RelevancyEvaluator`** replace the RAGAS dependency entirely for the two most important generation metrics.
5. The entire pipeline is composable: swap `VectorIndexRetriever` for `BM25LlamaRetriever` or wrap both in `QueryFusionRetriever` without touching generation or eval code.

### Retrieval quality

- The underlying embeddings and vector store are identical between v1 and v2 (same OpenAI model, same ChromaDB).
- **Recall@5 and MRR differences** are due to chunking strategy: v1 used Jaccard-based semantic chunks; v2 uses LlamaIndex `SentenceSplitter` (token-count-based).
- If you use the same chunk set in both (as `run_v2_retrieval_eval.py` does by loading the v1 JSONL into v2 nodes), results should be numerically identical for semantic search.

### When to choose each

| Scenario | Recommendation |
|---|---|
| Learning / debugging internals | **v1** — every step is explicit |
| Production / extensible app | **v2** — faster to extend, fewer bugs |
| Custom chunking (e.g. Jaccard semantic) | **v1** — full control |
| Plug-in rerankers, routers, agents | **v2** — LlamaIndex ecosystem |
| Evaluation harness | **v2** — built-in evaluators, less code |